# RTL-to-GDS walk-through: 4-bit counter on GF180MCU

Self-contained walk-through. Takes a Verilog counter all the way to a
manufacturable GDSII using LibreLane inside the
`hpretl/iic-osic-tools` Docker image. Every step shows the exact
command it runs.

**This notebook is shareable as a single file.** Drop it anywhere --
`~/Downloads`, `/tmp`, a USB stick, a fresh folder -- and it runs. All
state is written to a fixed host path under your HOME
(`~/eda/designs/`), not to the notebook's own folder, so the file
location has no effect on the flow.

**No extra Python packages.** Only stdlib + `subprocess` to drive
Docker. The counter RTL and the LibreLane `config.yaml` are embedded
inline, so the notebook has no external file dependencies.

**PDK.** We use the GF180MCU that `ciel` already installed inside the
image at `/foss/pdks/gf180mcuD/`. No external PDK fork is cloned --
for a bare block like this counter (no padring, no SRAM), the built-in
PDK is sufficient. The main deck shows when you'd add the wafer-space
fork for chip-level padring designs.

## What you need on the host

- Linux (x86_64 recommended), `~40 GB` free disk.
- Docker daemon running. Test with `docker ps`.
- Python 3.9+ for the notebook kernel itself. No other pip packages.

## What the walk-through covers

1. Pull the IIC-OSIC-TOOLS image (one-time, ~15 GB).
2. Start a headless container with a bind-mount into a host folder.
3. Write a tiny 4-bit counter `counter.v` into the project folder.
4. Write a minimal LibreLane `config.yaml` (inline).
5. Invoke `librelane` via `docker exec` and wait.
6. Grep `metrics.csv` for die area, timing, DRC/LVS.
7. Display the final-layout PNG that LibreLane auto-renders.

Each heavy step has a `RUN_*` flag at the top -- flip to `True` to
execute, leave `False` to just see the commands that would run. Use
this to rehearse the flow before committing to the 15 GB pull.

## Step 0 -- configuration

All paths, names, and `RUN_*` flags live here. Tweak, then run the
rest of the notebook top-to-bottom.

In [ ]:
from pathlib import Path
import os
import subprocess
import textwrap

# --- toggles: flip to True to actually execute a step. ---
RUN_PULL_IMAGE  = False   # ~15 GB one-time download
RUN_START_CTR   = False   # starts the long-running container
RUN_LIBRELANE   = False   # runs the flow (~1-2 min for the counter)

# --- container + image ---
IMAGE          = "hpretl/iic-osic-tools:next"
CONTAINER_NAME = "gf180"

# --- paths: HOST_* on your machine, CONTAINER_* inside the container ---
HOST_WORKSPACE    = Path.home() / "eda" / "designs"
HOST_PROJECT_DIR  = HOST_WORKSPACE / "counter_demo"
CONTAINER_WS      = "/foss/designs"
CONTAINER_PROJECT = f"{CONTAINER_WS}/counter_demo"

# --- GF180MCU PDK: built-in, shipped with the image by `ciel`.
# No external clone needed for a Classic-flow bare-block design.
CONTAINER_PDK_ROOT = "/foss/pdks"
PDK_NAME           = "gf180mcuD"
STD_CELL_LIB       = "gf180mcu_fd_sc_mcu7t5v0"


def run_or_print(cmd, do_it, *, shell_on_container=False, timeout=None):
    """Print the command; execute only if do_it is True.

    cmd: list of argv tokens, OR (when shell_on_container=True) a
    single bash script string to feed to `docker exec ... bash -lc`.
    """
    if shell_on_container:
        print("$ docker exec", CONTAINER_NAME, "bash -lc '<script>'")
        print(textwrap.indent(cmd, "  | "))
    else:
        print("$ " + " ".join(cmd))
    if not do_it:
        print("  (skipped -- flip the RUN_* flag to execute)\n")
        return None
    print("  (executing...)")
    args = (
        ["docker", "exec", CONTAINER_NAME, "bash", "-lc", cmd]
        if shell_on_container else cmd
    )
    proc = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    if proc.stdout.strip():
        print(proc.stdout[-4000:])
    if proc.returncode != 0 and proc.stderr.strip():
        print("STDERR (tail):")
        print(proc.stderr[-2000:])
    print(f"  returncode={proc.returncode}\n")
    return proc


HOST_WORKSPACE.mkdir(parents=True, exist_ok=True)
HOST_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Workspace on host: {HOST_WORKSPACE}")
print(f"Project dir:       {HOST_PROJECT_DIR}")
print(f"PDK (inside ctr):  {CONTAINER_PDK_ROOT}/{PDK_NAME}")

## Step 1 -- pull the toolchain image

One-time download. The IIC-JKU team rebuilds this image weekly with
LibreLane, Yosys, OpenROAD, KLayout, Magic, Netgen, ngspice and the
GF180MCU + IHP-SG13G2 PDKs already installed. Roughly 15 GB
compressed; budget 20+ GB after extraction.

Skip this if `docker images` already lists the image.

In [ ]:
run_or_print(["docker", "pull", IMAGE], RUN_PULL_IMAGE, timeout=1800)

## Step 2 -- start a long-running container

Headless container that stays alive so every subsequent cell can
`docker exec` into it.

Key flags:
- `-d` detached (background).
- `--name gf180` memorable handle.
- `-v HOST:CONTAINER:rw` bind-mount the workspace. Host edits show
  up inside; LibreLane artifacts land on the host.
- `--user $(id -u):$(id -g)` artifacts owned by you, not root.
- `--skip sleep infinity` is the image's built-in stay-alive entry
  point.

In [ ]:
start_cmd = [
    "docker", "run", "-d", "--name", CONTAINER_NAME,
    "-v", f"{HOST_WORKSPACE}:{CONTAINER_WS}:rw",
    "--user", f"{os.getuid()}:{os.getgid()}",
    IMAGE,
    "--skip", "sleep", "infinity",
]
run_or_print(start_cmd, RUN_START_CTR)

# Quick check (cheap, always runs):
subprocess.run(["docker", "ps", "--filter", f"name={CONTAINER_NAME}"])

## Step 3 -- write the counter RTL

A 4-bit synchronous up-counter with reset. 23 lines, embedded as a
Python string so this notebook has no external file dependency.

In [ ]:
COUNTER_V = """\
// 4-bit synchronous up-counter with active-high reset.
// Tiny on purpose: the full LibreLane flow runs in ~1-2 minutes.

module counter (
    input  wire       clk,
    input  wire       rst,
    output wire [3:0] q
);
    reg [3:0] cnt;

    always @(posedge clk) begin
        if (rst)
            cnt <= 4'b0;
        else
            cnt <= cnt + 4'b1;
    end

    assign q = cnt;
endmodule
"""

(HOST_PROJECT_DIR / "counter.v").write_text(COUNTER_V)
print(f"Wrote {HOST_PROJECT_DIR / 'counter.v'}")
print(COUNTER_V)

## Step 4 -- write the LibreLane config

Minimal GF180MCU `config.yaml` (Classic flow, no padring). Embedded
inline -- no template engine, no external package.

Key fields:
- `DESIGN_NAME`, `VERILOG_FILES` -- what to synthesize.
- `CLOCK_PORT`, `CLOCK_PERIOD` -- target timing (50 ns = 20 MHz).
- `DIE_AREA` -- absolute 300x300 um die.
- `PDN_*` -- straps and pitch for the power grid.
- `RUN_MAGIC_STREAMOUT: false` plus `substituting_steps.Magic.StreamOut:
  null`: Magic streamout has a Tcl regression in the current
  LibreLane/Magic combo; KLayout writes the final GDS.

In [ ]:
CONFIG_YAML = """\
# GF180MCU LibreLane config -- minimal walk-through.
meta:
  version: 3
  flow: Classic
  substituting_steps:
    Magic.StreamOut: null
    KLayout.XOR: null

RUN_MAGIC_STREAMOUT: false
RUN_KLAYOUT_XOR: false

DESIGN_NAME: counter
VERILOG_FILES:
  - dir::counter.v
CLOCK_PORT: clk
CLOCK_PERIOD: 50

# Die area: 300x300 um absolute (plenty of room for 4 flops).
FP_SIZING: absolute
DIE_AREA: [0.0, 0.0, 300.0, 300.0]

# Power / ground nets.
VDD_NETS:
  - VDD
GND_NETS:
  - VSS

# Signoff / streamout.
PRIMARY_GDSII_STREAMOUT_TOOL: klayout

# Routing / ESD.
DIODE_ON_PORTS: in
RT_MAX_LAYER: Metal4
PDN_MULTILAYER: false

# Power distribution network (straps + pitch).
PDN_VWIDTH: 5
PDN_HWIDTH: 5
PDN_VSPACING: 1
PDN_HSPACING: 1
PDN_VPITCH: 75
PDN_HPITCH: 75
PDN_EXTEND_TO: boundary

# Placement.
PL_TARGET_DENSITY_PCT: 65
MAX_FANOUT_CONSTRAINT: 10

# CTS defaults.
CTS_CLK_MAX_WIRE_LENGTH: 0
CTS_DISTANCE_BETWEEN_BUFFERS: 0
CTS_SINK_CLUSTERING_SIZE: 20
CTS_SINK_CLUSTERING_MAX_DIAMETER: 60

# Margins.
TOP_MARGIN_MULT: 1
BOTTOM_MARGIN_MULT: 1
LEFT_MARGIN_MULT: 6
RIGHT_MARGIN_MULT: 6
"""

config_path = HOST_PROJECT_DIR / "config.yaml"
config_path.write_text(CONFIG_YAML)
print(f"Wrote {config_path} ({len(CONFIG_YAML.splitlines())} lines)")
for i, line in enumerate(CONFIG_YAML.splitlines()[:25], 1):
    print(f"{i:>2}  {line}")
print("  ...")

## Step 5 -- run LibreLane

Inside the container we:
1. `cd` into the project dir (which lives in the bind-mount).
2. `source sak-pdk-script.sh gf180mcuD ...` to switch the container's
   active PDK to GF180 (the image defaults to IHP-SG13G2).
3. Run `librelane config.yaml` with `--pdk-root /foss/pdks` pointing
   at the built-in PDK collection that `ciel` populated when the image
   was built, and `--run-tag demo` so the output folder has a
   predictable name.

Expected wall time for the counter: 1-2 minutes.

In [ ]:
flow_script = textwrap.dedent(f"""
    set -euo pipefail
    cd {CONTAINER_PROJECT}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    librelane config.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {CONTAINER_PDK_ROOT} \\
        --manual-pdk \\
        --run-tag demo
""").strip()

run_or_print(flow_script, RUN_LIBRELANE,
             shell_on_container=True, timeout=900)

## Step 6 -- read the metrics

LibreLane writes `runs/demo/final/metrics.csv` (one metric per row).
We grep the essentials: die area, cell count, timing violations,
DRC / LVS, total power. A green chip has zeros across the violation
counters.

In [ ]:
import csv

metrics_path = HOST_PROJECT_DIR / "runs" / "demo" / "final" / "metrics.csv"
wanted = [
    "design__die__area__um2",
    "design__instance__count__stdcell",
    "timing__setup_vio__count",
    "timing__hold_vio__count",
    "magic__drc_error__count",
    "klayout__drc_error__count",
    "design__lvs_error__count",
    "power__total",
]

if not metrics_path.exists():
    print(f"metrics.csv not found: {metrics_path}")
    print("Did Step 5 complete? Set RUN_LIBRELANE = True and re-run.")
else:
    print(f"Reading {metrics_path}\n")
    found = {}
    with metrics_path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in wanted:
                found[row[0]] = row[1] if len(row) > 1 else ""
    for key in wanted:
        print(f"  {key:45s} {found.get(key, '(missing)')}")

## Step 7 -- show the final-layout render

LibreLane's `KLayout.Render` step already generated a PNG of the top
cell and copied it into `runs/demo/final/render/counter.png`. We just
display it here. For an interactive view of the GDS, open it on the
host:

```bash
klayout ~/eda/designs/counter_demo/runs/demo/final/gds/counter.gds
```

In [ ]:
from IPython.display import Image, display

png_path = HOST_PROJECT_DIR / "runs/demo/final/render/counter.png"
gds_path = HOST_PROJECT_DIR / "runs/demo/final/gds/counter.gds"

if png_path.exists():
    print(f"LibreLane render: {png_path}")
    display(Image(str(png_path)))
else:
    print(f"Render PNG not found: {png_path}")
    if gds_path.exists():
        print(f"But GDS is there: {gds_path}")
        print(f"  open it on the host with:  klayout {gds_path}")
    else:
        print("No GDS either -- Step 5 did not produce artifacts.")

## Cleanup and next steps

Host-side artifacts live in `~/eda/designs/counter_demo/`. They
survive stopping or removing the container.

```bash
# Stop the container (host artifacts untouched)
docker stop gf180

# Remove the container (image stays cached)
docker rm gf180

# Reclaim disk: purge the image (one-time ~15 GB)
docker image rm hpretl/iic-osic-tools:next
```

### Try next

- Edit `COUNTER_V` in Step 3: add an enable, widen to 8 bits, etc.
  Re-run from Step 3 onwards.
- Tighten `CLOCK_PERIOD` in Step 4 (e.g. 20 ns -> 50 MHz) and watch
  `timing__setup_vio__count` change in Step 6.
- Move to a full-chip padring design: follow the main deck
  `tutorials/rtl2gds-gf180-docker/main.pdf` which uses the
  wafer-space `gf180mcu-project-template` for a shuttle-ready chip.
  That's where the wafer-space PDK fork becomes necessary -- it ships
  the custom I/O cells (`gf180mcu_ws_io__dvdd`, `__dvss`) that the
  padring needs.